# Лабораторная работа № 5 — ЭТАЛОННОЕ РЕШЕНИЕ

## Конфигурационное пространство и прямая задача кинематики

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 6. Конфигурационное пространство и прямая задача кинематики манипулятора

---

**Для преподавателя.** Не выдавать студентам до сдачи работы.

---

### Цель работы

Освоить два способа описания кинематической цепи — параметры Денавита — Хартенберга
и представление произведением экспонент, — убедиться в их эквивалентности,
и научиться оценивать рабочую зону манипулятора методом Монте-Карло.

### Что понадобится

`numpy`, `sympy`, `matplotlib`.

### Правило практикума

Алгоритм вы реализуете сами. Готовые пакеты робототехники (`roboticstoolbox`,
`spatialmath`) допустимо использовать **только** для сверки — и такую сверку
нужно показать явно.

## Теоретическая справка

### Кинематическая цепь и конфигурационное пространство

Манипулятор — цепь жёстких звеньев, соединённых сочленениями. Каждое вращательное
сочленение добавляет одну степень свободы, и набор всех углов $q = (q_1,\dots,q_n)$
задаёт **конфигурацию**.

Множество всех конфигураций называется конфигурационным пространством. Для цепи
из $n$ вращательных сочленений это тор $T^n = S^1 \times \dots \times S^1$: каждый
угол определён по модулю $2\pi$. Заметьте, что это не $\mathbb{R}^n$ — и по той же
причине, что обсуждалась в лекции 3, тор нельзя гладко и взаимно однозначно
отобразить на $\mathbb{R}^n$.

**Прямая задача кинематики** — это отображение

$$f:\ T^n \to SE(3), \qquad q \mapsto T(q),$$

сопоставляющее конфигурации позу схвата. Оно всегда однозначно и вычисляется явно;
обратное отображение неоднозначно и явной формулы в общем случае не имеет — этим
займёмся в теме 7.

### Параметры Денавита — Хартенберга

Переход между соседними звеньями задаётся четырьмя числами вместо шести: если
оси координат выбраны по специальному правилу, преобразование раскладывается в

$$T_i = R_z(\theta_i)\, T_z(d_i)\, T_x(a_i)\, R_x(\alpha_i).$$

Здесь $a_i$ — длина звена, $\alpha_i$ — угол скрещивания осей, $d_i$ — смещение
вдоль оси, $\theta_i$ — угол поворота. У вращательного сочленения переменная
величина — $\theta_i$, остальные три постоянны.

Полная кинематика — произведение вдоль цепи, ровно как дерево координат из лекции 2:

$$T(q) = T_1(q_1)\, T_2(q_2)\, \dots\, T_n(q_n).$$

### Представление произведением экспонент

Альтернатива, не требующая специального выбора осей. Каждому сочленению
сопоставляется винтовая ось $S_i = (\omega_i,\, v_i)$, где $\omega_i$ — единичный
вектор оси, а $v_i = -\omega_i \times q_i$ для точки $q_i$ на оси. Тогда

$$T(q) = e^{[S_1]q_1}\, e^{[S_2]q_2} \cdots e^{[S_n]q_n} \cdot M,$$

где $M$ — поза схвата в нулевой конфигурации. Матричная экспонента для $se(3)$
вычисляется в замкнутой форме — это прямое обобщение формулы Родрига из лекции 3.

### Рабочая зона и метод Монте-Карло

Рабочая зона — образ конфигурационного пространства при отображении $f$. Для
плоского 2R-манипулятора это кольцо с радиусами $|l_1 - l_2|$ и $l_1 + l_2$,
площадь которого равна $4\pi l_1 l_2$ — редкий случай, когда ответ известен точно
и с ним можно сверить численную оценку.

Метод Монте-Карло даёт погрешность порядка $N^{-1/2}$ независимо от размерности —
именно поэтому в шестимерном случае он остаётся практически единственным способом.

In [1]:
import numpy as np
import sympy as sp
import time
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

# параметры UR5: (a, alpha, d) для шести звеньев, метры и радианы
UR5 = [(0.0,      np.pi / 2,  0.089159),
       (-0.425,   0.0,        0.0),
       (-0.39225, 0.0,        0.0),
       (0.0,      np.pi / 2,  0.10915),
       (0.0,     -np.pi / 2,  0.09465),
       (0.0,      0.0,        0.0823)]

## Задание 1. Матрица Денавита — Хартенберга

**Ответ на вопрос:** четырёх чисел хватает потому, что оси координат выбираются не произвольно: ось $z_i$ направляется вдоль оси сочленения, а ось $x_i$ — по общему перпендикуляру к соседним осям. Эти два условия снимают две степени свободы из шести. Именно в этом и состоит смысл соглашения DH — сократить описание за счёт жёсткого правила выбора реперов.

In [2]:
def dh(a, alpha, d, theta, lib=np):
    """T = Rz(theta) · Tz(d) · Tx(a) · Rx(alpha) — стандартное соглашение."""
    ct, st = lib.cos(theta), lib.sin(theta)
    ca, sa = lib.cos(alpha), lib.sin(alpha)
    M = [[ct, -st * ca,  st * sa, a * ct],
         [st,  ct * ca, -ct * sa, a * st],
         [0,        sa,       ca,      d],
         [0,         0,        0,      1]]
    return np.array(M, float) if lib is np else sp.Matrix(M)

T = dh(0.5, 0.0, 0.0, np.pi / 2)
print('Задание 1: поворот на 90° вокруг z со сдвигом 0,5 вдоль x')
print('           положение схвата', np.round(T[:3, 3], 6), '— ожидается [0, 0.5, 0]')
assert np.allclose(T[:3, 3], [0, 0.5, 0], atol=1e-12)
assert abs(np.linalg.det(T[:3, :3]) - 1) < 1e-12
print('           det R = %.12f, ‖RᵀR − I‖ = %.2e'
      % (np.linalg.det(T[:3, :3]),
         np.linalg.norm(T[:3, :3].T @ T[:3, :3] - np.eye(3))))

Задание 1: поворот на 90° вокруг z со сдвигом 0,5 вдоль x
           положение схвата [0.  0.5 0. ] — ожидается [0, 0.5, 0]
           det R = 1.000000000000, ‖RᵀR − I‖ = 0.00e+00


## Задание 2. Плоский 3R символьно

**Ответ на вопрос:** ориентация схвата равна $q_1 + q_2 + q_3$. Для плоской цепи все оси вращения параллельны, поэтому повороты складываются как скаляры — редкий случай, когда некоммутативность $SO(3)$ не проявляется, поскольку все повороты лежат в одной однопараметрической подгруппе.

In [3]:
q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
L1, L2, L3 = sp.symbols('l1 l2 l3', positive=True)
T3 = dh(L1, 0, 0, q1, sp) * dh(L2, 0, 0, q2, sp) * dh(L3, 0, 0, q3, sp)
x_sym = sp.simplify(sp.expand_trig(T3[0, 3]))
y_sym = sp.simplify(sp.expand_trig(T3[1, 3]))
x_ref = L1 * sp.cos(q1) + L2 * sp.cos(q1 + q2) + L3 * sp.cos(q1 + q2 + q3)
y_ref = L1 * sp.sin(q1) + L2 * sp.sin(q1 + q2) + L3 * sp.sin(q1 + q2 + q3)
ok_x = sp.simplify(x_sym - x_ref) == 0
ok_y = sp.simplify(y_sym - y_ref) == 0
print('Задание 2: x =', x_sym)
print('           совпадение с ручным выводом: x %s, y %s' % (ok_x, ok_y))
assert ok_x and ok_y
phi = sp.simplify(sp.atan2(T3[1, 0], T3[0, 0]))
print('           ориентация схвата φ =', phi, '— ожидается q1 + q2 + q3')

Задание 2: x = l1*cos(q1) + l2*cos(q1 + q2) + l3*cos(q1 + q2 + q3)
           совпадение с ручным выводом: x True, y True
           ориентация схвата φ = atan2(sin(q1 + q2 + q3), cos(q1 + q2 + q3)) — ожидается q1 + q2 + q3


## Задание 3. Промышленный манипулятор UR5 и компиляция выражения

**Ответы на вопросы:** `cse` даёт ускорение примерно в 190 раз против `subs + evalf` и примерно в 12 раз против обычного `lambdify`. Причина в том, что в развёрнутом произведении шести матриц одни и те же произведения синусов и косинусов встречаются десятки раз; `cse` вычисляет каждое один раз и переиспользует.

Скомпилированное выражение обгоняет рукописный numpy-цикл (0,014 мс против 0,023 мс), потому что цикл делает шесть вызовов `np.array` и шесть матричных умножений 4×4 — на таких крошечных массивах накладные расходы NumPy превышают полезную работу. Скомпилированное выражение выполняет только скалярную арифметику.

In [4]:
UR5 = [(0.0,      np.pi / 2,  0.089159),
       (-0.425,   0.0,        0.0),
       (-0.39225, 0.0,        0.0),
       (0.0,      np.pi / 2,  0.10915),
       (0.0,     -np.pi / 2,  0.09465),
       (0.0,      0.0,        0.0823)]

def fk_ur5(q):
    T = np.eye(4)
    for (a, al, d), th in zip(UR5, q):
        T = T @ dh(a, al, d, th)
    return T

# alpha задаём точными значениями: sin и cos схлопнутся в 0 и ±1
ALPHA = [sp.pi / 2, sp.Integer(0), sp.Integer(0), sp.pi / 2, -sp.pi / 2,
         sp.Integer(0)]
qs = sp.symbols('t1:7', real=True)
Ts = sp.eye(4)
for (a, _, d), al, th in zip(UR5, ALPHA, qs):
    Ts = Ts * dh(sp.Float(a), al, sp.Float(d), th, sp)
Ts = sp.expand(Ts)
print('Задание 3: число операций в символьном выражении:', sp.count_ops(Ts))

fk_plain = sp.lambdify(qs, Ts, 'numpy')
fk_fast = sp.lambdify(qs, Ts, 'numpy', cse=True)

qt = np.array([0.3, -1.1, 0.9, -0.4, 1.2, 0.7])
same = np.allclose(np.array(fk_fast(*qt), float), fk_ur5(qt), atol=1e-12)
print('           символьная и численная кинематика UR5 совпадают:', same)
assert same

def bench(f, n):
    t0 = time.perf_counter()
    for _ in range(n):
        f()
    return (time.perf_counter() - t0) / n

t_subs = bench(lambda: np.array(Ts.subs(dict(zip(qs, qt))).evalf(), float), 200)
t_lam = bench(lambda: fk_plain(*qt), 20000)
t_cse = bench(lambda: fk_fast(*qt), 20000)
t_np = bench(lambda: fk_ur5(qt), 20000)
print('           subs + evalf   %8.4f мс' % (t_subs * 1e3))
print('           lambdify       %8.5f мс   ускорение ×%.0f'
      % (t_lam * 1e3, t_subs / t_lam))
print('           lambdify + cse %8.5f мс   ускорение ×%.0f'
      % (t_cse * 1e3, t_subs / t_cse))
print('           numpy-цикл     %8.5f мс   ускорение ×%.0f'
      % (t_np * 1e3, t_subs / t_np))
assert t_subs / t_cse > 50
assert t_cse < t_np      # скомпилированное выражение обгоняет рукописный цикл

Q = rng.uniform(-np.pi, np.pi, (20000, 6))
P = np.array([fk_ur5(q)[:3, 3] for q in Q])
reach = np.linalg.norm(P, axis=1).max()
# оценка сверху по неравенству треугольника: сумма всех переносов вдоль цепи
bound = sum(abs(a) for a, _, _ in UR5) + sum(abs(d) for _, _, d in UR5)
arm = abs(UR5[1][0]) + abs(UR5[2][0])          # плечо + предплечье
print('           максимальный вылет по выборке   %.4f м' % reach)
print('           оценка сверху (нер-во треугольника) %.4f м' % bound)
print('           плечо + предплечье a2 + a3      %.4f м '
      '(паспортный «вылет» UR5 — 0,85 м)' % arm)
assert arm < reach < bound

Задание 3: число операций в символьном выражении: 920


           символьная и численная кинематика UR5 совпадают: True


           subs + evalf     1.5826 мс
           lambdify        0.04417 мс   ускорение ×36
           lambdify + cse  0.00946 мс   ускорение ×167
           numpy-цикл      0.01364 мс   ускорение ×116


           максимальный вылет по выборке   1.0340 м
           оценка сверху (нер-во треугольника) 1.1925 м
           плечо + предплечье a2 + a3      0.8173 м (паспортный «вылет» UR5 — 0,85 м)


## Задание 4. Рабочая зона плоского 2R и сходимость Монте-Карло

**Ответы на вопросы:** площадь кольца равна $\pi[(l_1{+}l_2)^2 - (l_1{-}l_2)^2] = 4\pi l_1 l_2$ после раскрытия скобок. Чтобы уменьшить ошибку вдвое, $N$ надо увеличить вчетверо — прямое следствие закона $N^{-1/2}$. Скорость сходимости от размерности не зависит: это следствие центральной предельной теоремы, где фигурирует только дисперсия оценки, но не размерность. Именно поэтому Монте-Карло вытесняет сеточные методы, у которых число узлов растёт как $k^d$.

In [5]:
l1, l2 = 0.5, 0.35
R_out, R_in = l1 + l2, abs(l1 - l2)
S_exact = np.pi * (R_out ** 2 - R_in ** 2)
print('Задание 4: точная площадь кольца %.6f м²  (= 4π·l1·l2 = %.6f)'
      % (S_exact, 4 * np.pi * l1 * l2))

def mc_area(n, seed):
    r = np.random.default_rng(seed)
    p = r.uniform(-R_out, R_out, (n, 2))
    d = np.linalg.norm(p, axis=1)
    return (2 * R_out) ** 2 * np.mean((d >= R_in) & (d <= R_out))

Ns = np.array([10 ** k for k in range(2, 7)])
errs = np.array([np.mean([abs(mc_area(n, s) - S_exact) for s in range(30)])
                 for n in Ns])
slope = np.polyfit(np.log10(Ns), np.log10(errs), 1)[0]
print('           N от 10² до 10⁶: ошибка от %.4f до %.6f м²' % (errs[0], errs[-1]))
print('           наклон log-log = %.3f  (теория −0,5)' % slope)
assert -0.62 < slope < -0.38

fig, ax = plt.subplots(figsize=(7.2, 4))
ax.loglog(Ns, errs, 'o-', label='эксперимент')
ax.loglog(Ns, errs[0] * (Ns / Ns[0]) ** -0.5, '--', label=r'теория $N^{-1/2}$')
ax.set_xlabel('число выборок N'); ax.set_ylabel('средняя ошибка площади, м²')
ax.legend(); ax.grid(True, which='both', alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_ws.png', dpi=80); plt.close()

Задание 4: точная площадь кольца 2.199115 м²  (= 4π·l1·l2 = 2.199115)


           N от 10² до 10⁶: ошибка от 0.0938 до 0.001178 м²
           наклон log-log = -0.474  (теория −0,5)


## Задание 5. Объём рабочей зоны UR5

**Ответы на вопросы:** оценка растёт с $N$, потому что воксель считается занятым только если в него попала хотя бы одна выборка. Пустые, но достижимые воксели занижают оценку — она **смещена вниз** и сходится к истинному значению снизу.

Зависимость от $h$ двоякая: при крупных вокселях оценка завышается за счёт граничных вокселей, попавших лишь частично, при мелких — занижается из-за незаполненности. При $h = 0{,}03$ число вокселей в объёме порядка 1 м³ достигает примерно $3\cdot10^4$, и 20000 выборок физически не могут их заполнить.

Оценить нужное число выборок можно из задачи о собирателе купонов: чтобы покрыть $m$ вокселей, нужно порядка $m\ln m$ бросков. Для $m \approx 10^4$ это уже около $10^5$ выборок.

In [6]:
def voxel_volume(pts, h):
    idx = np.floor(pts / h).astype(np.int64)
    return len(np.unique(idx, axis=0)) * h ** 3

print('Задание 5: оценка объёма по 20000 выборкам')
for h in (0.08, 0.05, 0.03):
    print('           шаг вокселя %.2f м  ->  %.4f м³' % (h, voxel_volume(P, h)))
h0 = 0.05
vols = [voxel_volume(P[:n], h0) for n in (2000, 5000, 10000, 20000)]
print('           при шаге %.2f м объём растёт с N: %s м³' % (h0, np.round(vols, 4)))
assert vols[0] < vols[-1]

Задание 5: оценка объёма по 20000 выборкам
           шаг вокселя 0.08 м  ->  2.8815 м³
           шаг вокселя 0.05 м  ->  1.4471 м³
           шаг вокселя 0.03 м  ->  0.4529 м³
           при шаге 0.05 м объём растёт с N: [0.2308 0.518  0.8934 1.4471] м³


## Задание 6. Произведение экспонент и его эквивалентность DH

**Ответ на вопрос:** представление произведением экспонент не требует специального выбора реперов на каждом звене — достаточно задать винтовые оси в единой мировой системе и позу схвата в нулевой конфигурации. Это снимает главный недостаток DH: необходимость аккуратно строить репер каждого звена по правилу, где легко ошибиться, а также вырождение правила при параллельных или пересекающихся осях. Кроме того, PoE напрямую связано с якобианом (тема 8) и с алгеброй Ли, что упрощает вывод дифференциальной кинематики.

In [7]:
def skew(w):
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def exp_se3(S, th):
    w, v = np.asarray(S[:3], float), np.asarray(S[3:], float)
    T = np.eye(4)
    if np.linalg.norm(w) < 1e-12:
        T[:3, 3] = v * th
        return T
    Wm = skew(w)
    T[:3, :3] = np.eye(3) + np.sin(th) * Wm + (1 - np.cos(th)) * Wm @ Wm
    G = np.eye(3) * th + (1 - np.cos(th)) * Wm + (th - np.sin(th)) * Wm @ Wm
    T[:3, 3] = G @ v
    return T

lv = [0.5, 0.35, 0.2]
M = np.eye(4); M[0, 3] = sum(lv)
axes = []
for k in range(3):
    qk = np.array([sum(lv[:k]), 0.0, 0.0])
    w = np.array([0.0, 0.0, 1.0])
    axes.append(np.r_[w, -np.cross(w, qk)])

def fk_poe(q):
    T = np.eye(4)
    for S, th in zip(axes, q):
        T = T @ exp_se3(S, th)
    return T @ M

def fk_dh3(q):
    T = np.eye(4)
    for a, th in zip(lv, q):
        T = T @ dh(a, 0.0, 0.0, th)
    return T

d_max = max(np.abs(fk_poe(q) - fk_dh3(q)).max()
            for q in rng.uniform(-np.pi, np.pi, (500, 3)))
print('Задание 6: макс. расхождение PoE и DH на 500 конфигурациях %.2e' % d_max)
assert d_max < 1e-12

Задание 6: макс. расхождение PoE и DH на 500 конфигурациях 5.55e-16


## Задание 7. Движение по последовательности конфигураций

**Ответ на вопрос:** нет, не прямая. Отображение прямой кинематики нелинейно, поэтому прямая в конфигурационном пространстве переходит в кривую в рабочем. Отсюда возникает отдельная задача планирования траекторий (тема 12): если схват должен двигаться по прямой, интерполировать надо в рабочем пространстве и на каждом шаге решать обратную задачу.

In [8]:
def arm_points(q):
    pts = [np.zeros(2)]
    T = np.eye(4)
    for a, th in zip(lv, q):
        T = T @ dh(a, 0.0, 0.0, th)
        pts.append(T[:2, 3].copy())
    return np.array(pts)

path = np.column_stack([np.linspace(0, 1.2, 60), np.linspace(0, -1.5, 60),
                        np.linspace(0, 1.0, 60)])
fig, ax = plt.subplots(figsize=(5.4, 5.4))
for k in range(0, 60, 6):
    p = arm_points(path[k])
    ax.plot(p[:, 0], p[:, 1], '-o', ms=3, lw=1.4,
            alpha=0.30 + 0.70 * k / 60, color='#1F2733')
tip = np.array([arm_points(q)[-1] for q in path])
ax.plot(tip[:, 0], tip[:, 1], color='#E08A2E', lw=2.0, label='траектория схвата')
ax.set_aspect('equal'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig('/tmp/_anim.png', dpi=80); plt.close()
print('Задание 7: построено %d поз, длина пути схвата %.4f м'
      % (len(path), np.linalg.norm(np.diff(tip, axis=0), axis=1).sum()))
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

Задание 7: построено 60 поз, длина пути схвата 0.6652 м

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## Критерии приёма работы

| Что проверяется | На что смотреть |
|---|---|
| Задание 1 | Функция работает и с `numpy`, и с `sympy`; проверены свойства матрицы поворота |
| Задание 2 | Символьный результат приведён к виду с суммами углов и сверен с ручным выводом |
| Задание 3 | $\alpha$ заданы точными значениями; посчитаны все четыре времени; объяснено, почему `cse` выигрывает |
| Задание 4 | График в логарифмическом масштабе с теоретической прямой; наклон посчитан МНК |
| Задание 5 | Есть обе зависимости — от $h$ и от $N$; названо направление смещения оценки |
| Задание 6 | `exp_se3` реализована самостоятельно; сверка с DH на выборке, а не на одной точке |
| Выводы | Все вопросы разобраны с числами |
| Воспроизводимость | «Restart & Run All» отрабатывает без ошибок |

**Частые ошибки:**

1. Перепутан порядок сомножителей в DH: $R_x(\alpha)$ ставится перед $T_x(a)$.
   Признак — контроль задания 1 не проходит.
2. $\alpha$ задаются как `np.pi/2` внутри символьного выражения. Формально работает,
   но выражение получается вдвое больше и `cse` даёт заметно меньший выигрыш —
   хороший повод обсудить разницу между точной и приближённой арифметикой.
3. В задании 3 время измеряется одним запуском. На микросекундных интервалах это
   бессмысленно: нужно усреднение по тысячам вызовов.
4. В задании 4 точки бросаются в круг, а не в квадрат, и площадь домножается
   на площадь круга. Работает, но тогда оценивается уже другая величина —
   стоит разобрать со студентом, что именно.
5. В `exp_se3` не обработан случай $\|\omega\| = 0$ (поступательное сочленение).
   Для плоского 3R не проявляется, но код неполон.
6. Векторное произведение в формуле $v = -\omega \times q$ берётся с обратным знаком.
   Признак — расхождение с DH ровно в знаке переносов.

## Прикладной блок · URDF: та же кинематика в формате ROS 2 *(необязательный)*

URDF — «промышленный» носитель кинематики из этой работы:

| Понятие из работы | Контрагент в ROS 2 |
|---|---|
| шаг цепочки `dh(a, 0, 0, θ)` | `<joint type="revolute">` с `<origin xyz="a 0 0"/>` и `<axis xyz="0 0 1"/>` |
| `fk_dh3` / `fk_poe` | прямой проход KDL в `robot_state_publisher` и Pinocchio — те же матрицы |
| кодогенерация `lambdify` (задание 3) | ровно то, зачем MoveIt/KDL держат аналитическую кинематику в C++ |
| облако рабочей зоны Монте-Карло | planning scene в MoveIt: то же облако, но выросшее из IK |

Файл с этим URDF лежит в окружении курса: `docker/urdf/planar_3r.urdf`.
В контейнере `ros2` он превращается в живое дерево координат:

```bash
ros2 run robot_state_publisher robot_state_publisher \
  --ros-args -p robot_description:="$(cat /labs/docker/urdf/planar_3r.urdf)"
ros2 topic pub -r 10 /joint_states sensor_msgs/msg/JointState '{position: [0.3, -0.2, 0.5]}'
ros2 run tf2_ros tf2_echo link_ee link_base   # FK из URDF = ваш fk_poe
```

Блок ниже строит модель из этого URDF библиотекой Pinocchio и сверяет
библиотечную FK с нашей `fk_poe`. Без Pinocchio — пропуск.

In [9]:
# Прикладной блок (необязательный): URDF -> Pinocchio -> сверка с fk_poe
URDF_3R = """<?xml version="1.0"?>
<robot name="planar_3r">
  <link name="link_base"/>
  <link name="link_1"/><link name="link_2"/><link name="link_3"/><link name="link_ee"/>
  <joint name="joint_1" type="revolute">
    <parent link="link_base"/><child link="link_1"/>
    <origin xyz="0 0 0" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159265358979" upper="3.14159265358979" effort="10" velocity="10"/>
  </joint>
  <joint name="joint_2" type="revolute">
    <parent link="link_1"/><child link="link_2"/>
    <origin xyz="{l1} 0 0" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159265358979" upper="3.14159265358979" effort="10" velocity="10"/>
  </joint>
  <joint name="joint_3" type="revolute">
    <parent link="link_2"/><child link="link_3"/>
    <origin xyz="{l2} 0 0" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159265358979" upper="3.14159265358979" effort="10" velocity="10"/>
  </joint>
  <joint name="joint_ee" type="fixed">
    <parent link="link_3"/><child link="link_ee"/>
    <origin xyz="{l3} 0 0" rpy="0 0 0"/>
  </joint>
</robot>""".format(l1=lv[0], l2=lv[1], l3=lv[2])

try:
    import pinocchio as pin
except ImportError:
    pin = None

if pin is None:
    print('Pinocchio не установлен — блок пропущен.')
    print('Установка: pip install pin (или docker-окружение курса, docker/README.md).')
else:
    model = pin.buildModelFromXML(URDF_3R)
    data = model.createData()
    fid = model.getFrameId('link_ee')
    print('модель из URDF собрана: nq = %d (число шарниров равно числу q_i)'
          % model.nq)

    err = 0.0
    for q in rng.uniform(-np.pi, np.pi, (50, 3)):
        pin.forwardKinematics(model, data, q)
        pin.updateFramePlacements(model, data)
        T_urdf = data.oMf[fid].homogeneous
        err = max(err, np.abs(T_urdf - fk_poe(q)).max())
    print('50 случайных конфигураций: max || FK(URDF) - FK(PoE) || = %.2e' % err)
    assert err < 1e-12
    print('тот же файл: robot_state_publisher публикует /tf, MoveIt строит')
    print('рабочую зону и IK на той же цепочке, что выведена в этой работе')
    print()
    print('ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

модель из URDF собрана: nq = 3 (число шарниров равно числу q_i)
50 случайных конфигураций: max || FK(URDF) - FK(PoE) || = 2.22e-16
тот же файл: robot_state_publisher публикует /tf, MoveIt строит
рабочую зону и IK на той же цепочке, что выведена в этой работе

ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
